In [ ]:
# Preliminary
# Set environment variables - not mandatory but recommended if you use parallel with maximum number of CPUs
#import os
#os.environ['OPENBLAS_NUM_THREADS'] = '1'
#os.environ['MKL_NUM_THREADS'] = '1'

import ray
ray.init(num_cpus=4)

In [ ]:
import wannierberri as wberri
print (f"Using WannierBerri version {wberri.__version__}")
import numpy as np
import scipy
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
# Importing data from wannier90
wandata = wberri.WannierData.from_w90_files(seedname='Fe', files=['mmn', 'eig', 'chk', 'uHu'])

# Create the system object
system=wberri.System_R.from_wannierdata(wandata=wandata,
                        berry=True,   # needed to calculate "external terms" of Berry connection or curvature , reads ".mmn" file\
                        morb=True,    # needed to calculate orbital magnetization , reads ".uHu" file\
                        )

# Setting the pointgroup from the symmetry operations
from irrep.spacegroup import SpaceGroup
spacegroup = SpaceGroup.from_cell(real_lattice=system.real_lattice,
                                  positions=[[0,0,0]],   # only 1 Fe atoms at origin
                                  typat=[1],    # atomic number is not important here
                                  spinor=True,
                                  magmom=[[0,0,1]],   # magnetic moment along z
                                  include_TR=True,   # include symmetries that involve time-reversal (TR)
                                                      # operations, but preserve the spins,
                                                      # like TR*C2x, but not TR alone
                              )

system.set_pointgroup(spacegroup=spacegroup)

# Alternatively, one can set the pointgroup from a list of generators, but one needs to be careful, not to
# include symmetries that are not present in the system,
# generators=["Inversion","C4z","TimeReversal*C2x"]
# system.set_pointgroup(symmetry_gen=generators)

In [ ]:
# You can save wannier data to .npz files for future uses
wandata.to_npz("Fe_wannierdata_npz")
system.to_npz("Fe_system_npz")

wandata_loaded = wberri.WannierData.from_npz("Fe_wannierdata_npz", files=['mmn', 'eig', 'chk', 'uHu'])
system_loaded = wberri.System_R.from_npz("Fe_system_npz")

In [ ]:
# all kpoints given in reduced coordinates
path=wberri.Path.from_nodes(system,
                 nodes=[
                    [0.0000, 0.0000, 0.0000 ],   #  G
                    [0.500 ,-0.5000, -0.5000],   #  H
                    [0.7500, 0.2500, -0.2500],   #  P
                    [0.5000, 0.0000, -0.5000],   #  N
                    [0.0000, 0.0000, 0.000  ] ] , #  G
                 labels=["G","H","P","N","G"],
                 length=200 )   # length [ Ang] ~= 2*pi/dk

# uncomment some of these lines to see what path you have generated
# print (path)
# print (path.getKline())
# for K in path.get_K_list():
#   print(K)

# print(np.array([K.K for K in path.get_K_list()]))

In [ ]:
# Make tabulators for wanted quantities in k-path
tabulators = { "energy": wberri.calculators.tabulate.Energy(),  # Interpolate band energies
               "berry_curvature" : wberri.calculators.tabulate.BerryCurvature(),   # Interpolate Berry curvature
             }

tab_all_path = wberri.calculators.TabulatorAll(
                    tabulators,
                    ibands = np.arange(0,18),
                    mode = "path"
                    )

In [ ]:
# Run calculation along k-path
result=wberri.run(system,
                  grid=path,
                  calculators = {"tabulate" : tab_all_path},
                  print_Kpoints = False)

print (result.results)
path_result = result.results["tabulate"]

In [ ]:
# plot the bands and compare with QuantumEspresso
EF = 17.5502

# Import the pre-computed bands from quantum espresso
A = np.loadtxt(open("Fe.bands.dat.gnu","r"))
alatt = 2.86
A[:,0] *= 2*np.pi/alatt
A[:,1] -= EF
# plot it as dots
plt.scatter(A[:,0], A[:,1], s=5, label="QE")

# plot interpolated bands
path_result.plot_path_fat(path,
              quantity=None,
              save_file="Fe_bands+QE.pdf",
              Eshift=EF,
              Emin=-10, Emax=35,
              iband=None,
              mode="fatband",
              fatfactor=20,
              cut_k=False,
              close_fig=False,
              show_fig=False,
              label = "WB"
              )

plt.legend()
plt.show()
plt.close()

In [ ]:
# plot the Berry curvature
path_result.plot_path_fat( path,
              quantity='berry_curvature',
              component='z',
              save_file=None, #"Fe_bands+berry.pdf",
              Eshift=0,
              Emin=4,  Emax=25,
              iband=None,
              mode="fatband",
              fatfactor=4,
              cut_k=False,
              close_fig=False,
              show_fig=False,
              )
plt.show()
plt.close()

# The size of the dots corresponds to the magnitude of BC on a logarithmic scale

In [ ]:
calculators = {}
Efermi = np.linspace(17,18,101)

# Set a grid
grid = wberri.Grid(system, length=50)   # length [ Ang] ~= 2*pi/dk

calculators ["ahc"] = wberri.calculators.static.AHC(Efermi=Efermi)
calculators ["morb"] = wberri.calculators.static.Morb(Efermi=Efermi)

result_run = wberri.run(system,
            grid=grid,
            calculators = calculators,
            adpt_num_iter=5,
            fout_name='Fe',
            restart=False,
            allow_restart=True,
            )

In [ ]:
#plot results from different iterations
#plot results from different iterations
for i in range(5):
    res = np.load(f"Fe-ahc_iter-{i:04d}.npz")
    # print (list(res.keys()))
    ef = res["Energies_0"]
    ahc_xy = res["data"][:,2]
    # alternatively from text files:
    # a = np.loadtxt(f"Fe-ahc_iter-{i:04d}.dat")
    # ef = a[:,0]
    # ahc_xy = a[:,3]
    plt.plot(ef,ahc_xy,label = f"iteration-{i}")
#plt.ylim(-1000,1000)
plt.xlabel("E [eV]")
plt.ylabel("AHC (S/m)")
plt.legend()
plt.show()

In [ ]:
#plot results from different iterations
#plot results from different iterations
for i in range(5):
    res = np.load(f"Fe-morb_iter-{i:04d}.npz")
    # print (list(res.keys()))
    ef = res["Energies_0"]
    morb_z = res["data"][:,2]
    # alternatively from text files:
    # a = np.loadtxt(f"Fe-ahc_iter-{i:04d}.dat")
    # ef = a[:,0]
    # ahc_xy = a[:,3]
    plt.plot(ef,morb_z,label = f"iteration-{i}")
#plt.ylim(-1000,1000)
plt.xlabel("E [eV]")
plt.ylabel("Morb (mu_B)")
plt.legend()
plt.show()